In [ ]:
#!/usr/bin/env python
# coding: utf-8
"""
Zaliapin-Ben-Zion Earthquake Network — Italy INGV Catalog (1985-2025)

Run as a script  : python ITALY_network_ZBZ.py
Convert to notebook: python convert_to_notebook.py ITALY_network_ZBZ.py notebooks/ITALY_network_ZBZ.ipynb
"""

import logging
import pickle
import time
from pathlib import Path

import matplotlib.pyplot as plt
import networkx as nx
import numpy as np
import pandas as pd
import plotly.express as px
import plotly.io as pio
import seaborn as sns

from src.network import build_baiesi_paczuski_network, build_zaliapin_ben_zion_network
from src.metrics import estimate_gamma_mle, test_power_law, measure_pa_forest, plot_preferential_attachment
from src.assortativity import compute_assortativity, plot_assortativity, plot_knn, plot_rich_club
from src.centrality import (
    plot_centrality_correlation,
    plot_top_n_cells,
    plot_geo_top_n_interactive,
    plot_geo_centrality_overlap,
    compute_bb_fitness_events,
    plot_bb_fitness,
    plot_bb_fitness_theory,
    plot_bb_fitness_geo,
)
from src.community import (
    run_louvain,
    run_consensus_louvain,
    run_spectral,
    run_infomap,
    run_directed_louvain,
    run_hdbscan_spectral,
    run_hdbscan_geo,
    run_bigclam,
    compare_partitions,
    plot_partition_scores,
    compute_nmi_matrix,
    plot_nmi_heatmap,
    plot_community_geo,
)
from src.viz import (
    analyze_degree_distribution,
    analyze_degree_distribution_log_binning,
    analyze_in_out_degree_distribution,
    plot_ccdf_with_fit,
    plot_degree_distribution_linear,
)
from src.plotutils import setup_matplotlib, configure_saves, savefig, save_plotly

try:
    from IPython.display import display
except ImportError:
    display = print  # type: ignore[assignment]

logging.basicConfig(level=logging.INFO, format="%(levelname)s  %(message)s")
sns.set_theme(style="whitegrid")
pio.renderers.default = "notebook"

DATA_DIR    = Path("data/INGV")
RESULTS_DIR = Path("results")
RESULTS_DIR.mkdir(exist_ok=True)
(RESULTS_DIR / "data").mkdir(exist_ok=True)
(RESULTS_DIR / "cache").mkdir(exist_ok=True)
(RESULTS_DIR / "gephi").mkdir(exist_ok=True)

CUT_YEAR   = 1985
TARGET_CRS = "epsg:32632"  # UTM Zone 32N — Italy

# ZBZ model hyperparameters (same metric as BP; Zaliapin et al. 2008, PRL 101, 018501)
ZBZ_B          = 1.0    # Gutenberg-Richter b-value
ZBZ_ALPHA      = 1.0    # Omori time exponent
ZBZ_D          = 1.6    # fractal dimension of epicentre distribution
ZBZ_T_MAX_DAYS = 730.0  # look-back window (2 years)
ZBZ_ETA_0      = None   # None = auto-detect threshold via GMM trough

SAVE_PDF: bool = True
SAVE_JPG: bool = True

setup_matplotlib()
configure_saves(SAVE_JPG, SAVE_PDF, RESULTS_DIR / "figures" / "italy" / "zbz")

## Data Loading

The INGV catalog covers Italian seismicity 1985–2025 at M ≥ 1.5.  Events are
filtered to `year ≥ 1985` (catalog completeness threshold) and sorted by UTC
origin time.  The integer reset index becomes the node ID used in both the BP
metric computation and the final ZBZ forest.

The ZBZ model begins with the same catalog as BP but adds a statistical
**declustering** step: not all events become nodes in the final network —
background events become roots while aftershocks retain their parent links.
The same catalog row ordering is preserved throughout, so ZBZ node IDs are
directly comparable to BP node IDs.

**Expected output:** ≈ 215,000 rows, RAM ≈ 30–60 MB.  The head/tail table
confirms chronological ordering and presence of all required columns.

In [ ]:
print("Loading Italy earthquake catalog...")
df = pd.read_csv(DATA_DIR / "italy_earthquakes_1985_2025.csv")
df["time"] = pd.to_datetime(df["time"], utc=True)
df_net = (
    df[df["time"].dt.year >= CUT_YEAR]
    .sort_values("time")
    .reset_index(drop=True)
)

print(f"Loaded {len(df_net):,} earthquakes "
      f"({df_net['time'].dt.year.min()}–{df_net['time'].dt.year.max()})")
print(f"RAM: {df_net.memory_usage(deep=True).sum() / 1024**2:.1f} MB")
display(pd.concat([df_net.head(3), df_net.tail(3)]))

## Network Construction

The **Zaliapin-Ben-Zion (ZBZ)** model extends BP by applying a statistical
threshold to decluster the catalog into **background** and **aftershock**
seismicity.  The construction proceeds in three steps:

**Step 1 — Nearest-neighbour distance.**  For each event $j$, compute the
BP proximity metric to all prior events within $t_{\max}$ and take the minimum:

$$\eta_j^* = \min_{i < j,\, t_{ij} \leq t_{\max}} n_{ij},
\quad n_{ij} = t_{ij}^{\alpha} \cdot r_{ij}^{d} \cdot 10^{-b\, m_i}$$

The array $\log_{10} \eta_j^*$ (nearest-neighbour distances) has a characteristic
**bimodal distribution**: a left mode of aftershock pairs (small $t$, small $r$,
large $m_i$) and a right mode of background pairs (large $t$, large $r$, small $m_i$).

**Step 2 — GMM threshold.**  A 2-component Gaussian Mixture Model is fitted to
$\log_{10} \eta^*$.  The threshold $\eta_0$ is placed at the trough (minimum
density) between the two modes — the point that minimises misclassification
between the aftershock and background populations.

**Step 3 — Graph.**  Edge $i \to j$ from the full BP forest is kept iff
$\log_{10} \eta_j^* < \eta_0$.  Events with $\log_{10} \eta_j^* \geq \eta_0$
are classified as **background**: their BP parent link is removed and they
become roots of new aftershock trees.  The result is a **sparser, seismologically
meaningful forest** where each tree is an aftershock family rooted at a
background mainshock.

*Reference:* Zaliapin, I., Gabrielov, A., Keilis-Borok, V., & Wong, H. (2008).
Clustering analysis of seismicity and aftershock identification.
*Physical Review Letters*, 101(1), 018501.

**Expected output:** BP metric computation ≈ 5–10 min; bimodal histogram
should show two clearly separated peaks at roughly $\log_{10}\eta \approx -6$
(aftershock mode) and $\log_{10}\eta \approx -2$ (background mode).  The GMM
threshold $\eta_0$ typically falls between −4 and −2.  After thresholding:
≈ 10,000–40,000 background roots (5–20% of catalog); the same number of trees.

In [ ]:
print(f"Computing BP nearest-neighbour distances "
      f"(b={ZBZ_B}, α={ZBZ_ALPHA}, d={ZBZ_D}, t_max={ZBZ_T_MAX_DAYS:.0f} days)…")
t_build = time.time()
G_bp, log_eta_nn = build_baiesi_paczuski_network(
    df_net,
    b=ZBZ_B,
    alpha=ZBZ_ALPHA,
    d=ZBZ_D,
    t_max_days=ZBZ_T_MAX_DAYS,
    return_nn_distances=True,
)
print(f"BP metric done ({time.time() - t_build:.1f}s)")

# Plot bimodal log10(η) distribution
valid_eta = log_eta_nn[~np.isnan(log_eta_nn)]
fig, ax = plt.subplots(figsize=(10, 5))
ax.hist(valid_eta, bins=300, density=True, color="steelblue", alpha=0.65,
        label=r"$\log_{10}(\eta)$ distribution")
ax.set_xlabel(r"$\log_{10}(\eta)$  — nearest-neighbour distance", fontsize=12)
ax.set_ylabel("Density", fontsize=12)
ax.set_title("Bimodal nearest-neighbour distance distribution — ZBZ Italy", fontsize=13)
ax.grid(True, ls="--", alpha=0.3)
ax.legend(fontsize=11)
plt.tight_layout()
savefig("zbz_eta_distribution_italy")
plt.show()

print("Building ZBZ network (applying GMM threshold)…")
G, eta_0 = build_zaliapin_ben_zion_network(G_bp, log_eta_nn, eta_0=ZBZ_ETA_0)
print(f"ZBZ threshold η₀ = {eta_0:.3f}  (log₁₀ space)")
print(f"Build time total: {time.time() - t_build:.1f}s")
print(f"Nodes: {G.number_of_nodes():,}  Edges: {G.number_of_edges():,}")

## Tree Structure Verification

Unlike BP (which forces every event to have a parent, producing one spanning
tree for a dense continuous catalog), ZBZ allows background events to be roots.
The number of trees equals the number of background events — each is the root
of its own aftershock family.

**Background fraction** (roots / total) is the ZBZ estimate of spontaneous
seismicity not triggered by any identified parent within the look-back window.
For Italian seismicity, expected values are 5–20% (lower than sparse catalogs
because the dense INGV catalog resolves more aftershock pairs).

**Tree size distribution** follows a power law: most trees are small
(1–10 events, isolated background events with few immediate aftershocks), while
a few large trees correspond to the most energetic mainshock sequences
(L'Aquila 2009 → ~10,000 events; Amatrice 2016 → ~30,000 events).

**Expected output:** max in-degree = 1 (forest valid); roots ≈ 10,000–40,000
(5–20%); trees = same count as roots; top-10 tree sizes should reveal a
power-law gap between the largest sequence (thousands of events) and the
next tier (hundreds).

In [ ]:
in_degrees  = [d for _, d in G.in_degree()]
out_degrees = [d for _, d in G.out_degree()]

max_in   = max(in_degrees)
n_roots  = sum(1 for d in in_degrees  if d == 0)
n_leaves = sum(1 for d in out_degrees if d == 0)
n_trees  = nx.number_weakly_connected_components(G)

print(f"Max in-degree       : {max_in}  (must be ≤ 1 for a valid forest)")
print(f"Background roots    : {n_roots:,}  ({100*n_roots/G.number_of_nodes():.1f}% of catalog)")
print(f"Aftershock fraction : {100*(1 - n_roots/G.number_of_nodes()):.1f}%")
print(f"Leaves (out=0)      : {n_leaves:,}  ({100*n_leaves/G.number_of_nodes():.1f}%)")
print(f"Trees (WCCs)        : {n_trees:,}")

wcc_sizes = sorted([len(c) for c in nx.weakly_connected_components(G)], reverse=True)
print(f"\nTop-10 tree sizes: {wcc_sizes[:10]}")

## Hub Map — 2D

The top 100 events by out-degree are plotted geographically, coloured and
sized by out-degree.  Marker size is linearly scaled to [8, 35] pixels;
points are rendered in ascending out-degree order so the highest values
appear on top.

In the ZBZ model, **high out-degree ↔ productive background mainshock**:
events classified as background that directly triggered many aftershocks.
Unlike BP (where the single spanning tree forces all events to have a parent,
inflating out-degrees by connecting unrelated distant events), ZBZ hubs are
constrained to events with genuine, statistically significant aftershock families.

**Expected output:** hubs concentrated along active fault systems —
Apennine arc, Calabrian arc, and Friuli.  The highest out-degree events
should correspond to major documented sequences: L'Aquila 2009, Amatrice 2016,
Irpinia 1980.  The spatial pattern should be sharper and more geologically
interpretable than BP because spurious background-to-background links are removed.

In [ ]:
df_nodes = pd.DataFrame([
    {
        "event_idx":  n,
        "out_degree": G.out_degree(n),
        "lat":        G.nodes[n]["lat"],
        "lon":        G.nodes[n]["lon"],
        "magnitude":  G.nodes[n]["magnitude"],
    }
    for n in G.nodes()
])
df_hubs = df_nodes.nlargest(100, "out_degree").copy()
print(f"Top 100 hubs: out_degree range [{df_hubs['out_degree'].min():.0f}, {df_hubs['out_degree'].max():.0f}]")

df_hubs = df_hubs.sort_values("out_degree")  # ascending → high values on top in plotly

_od_min = df_hubs["out_degree"].min()
_od_max = df_hubs["out_degree"].max()
df_hubs["marker_size"] = 8 + 27 * (df_hubs["out_degree"] - _od_min) / max(_od_max - _od_min, 1)

_IT_BOUNDS = dict(west=3, east=22, south=34, north=48)
MAP_WIDTH  = 770
MAP_HEIGHT = 700

fig = px.scatter_map(
    df_hubs, lat="lat", lon="lon",
    color="out_degree", size="marker_size",
    size_max=35, opacity=0.85,
    color_continuous_scale="inferno",
    map_style="carto-positron",
    hover_name="event_idx",
    hover_data={"magnitude": ":.2f", "out_degree": True, "marker_size": False},
    title="ZBZ Network Hubs: Top 100 Most Productive Background Events — Italy",
)
fig.update_layout(
    margin={"r": 0, "t": 40, "l": 0, "b": 0},
    width=MAP_WIDTH, height=MAP_HEIGHT,
    map=dict(center={"lat": 41.9, "lon": 12.5}, zoom=0, bounds=_IT_BOUNDS),
)
save_plotly(fig, "zbz_hub_map_2d_italy")
fig.show()

## Out-Degree Distribution — Linear

Linear-scale bar chart of out-degree (direct aftershocks per background event),
truncated at $k = 50$, plotted on a log y-axis.

ZBZ out-degrees are **structurally cleaner** than BP: by removing background-to-
background links, the distribution more faithfully reflects the Omori-law
branching structure of genuine aftershock sequences.  The bulk (most background
events trigger 0–2 aftershocks) is steeper than in BP; the tail (mainshocks
with large aftershock families) should follow a cleaner power law.

**Expected output:** a steeply declining bar chart dominated by $k = 0$ and
$k = 1$; fewer events at large $k$ than in BP (because ZBZ removes the
many spurious long-range links that inflate BP out-degree for older events).

In [ ]:
plot_degree_distribution_linear(G, "ZBZ Italy", max_degree=50, weighted=False)

## Out-Degree Distribution — Log-Log

Log-log scatter of $P(k)$ vs $k$ for in-degree and out-degree separately.

ZBZ in-degree is $\{0, 1\}$ (forest property, same as BP).  Out-degree tail
exponent $\gamma$ should be similar to BP (same metric, same underlying
Omori-law branching structure) but with a cleaner power-law range because
background-to-background links that scatter the BP tail are removed.

**Expected output:** in-degree — two isolated points; out-degree — power-law
tail for $k \geq 2$ with $\gamma \approx 2.0$–$2.5$.  The scatter at high $k$
may be slightly less noisy than BP due to the sparser, more homogeneous forest.

In [ ]:
analyze_in_out_degree_distribution(G, "ZBZ Italy")
analyze_degree_distribution(G, "ZBZ Italy")

## Out-Degree Distribution — Log Binning

Logarithmic binning stabilises the sparse high-$k$ tail and provides a
visual estimate of the branching exponent $\gamma$ before the MLE step.
Bin width normalisation yields probability *density* $P(k)$, enabling
comparison of the slope across different total event counts (e.g., comparing
ZBZ directly to the denser BP tail).

**Expected output:** a clean power-law tail for $k \geq 2$, slope ≈ −2 to
−2.5, R² > 0.85.  Compared to TL/HVG log-binned plots, the ZBZ tail starts
at lower $k$ (out-degree minimum is 0, not 2) and extends further (mainshocks
with hundreds of direct aftershocks appear at the right end).

In [ ]:
analyze_degree_distribution_log_binning(G, "ZBZ Italy", k_min_fit=2)

## Out-Degree Distribution — CCDF

The CCDF $P(K \geq k)$ on a log-log scale avoids binning artefacts.
For a power-law out-degree with exponent $\gamma$, the CCDF has slope
$-(\gamma - 1)$.

**Expected output:** approximately linear CCDF for $k \geq 2$ with slope
≈ −1 to −1.5 (corresponding to $\gamma \approx 2$–$2.5$).  The ZBZ CCDF
should be visually cleaner than BP's at the tail because the threshold
removes the diffuse low-weight background links that inflate BP's high-$k$
region.

In [ ]:
plot_ccdf_with_fit(G, "ZBZ Italy", k_min_fit=2)

## MLE Gamma — Branching Exponent

Maximum-likelihood estimate of the branching exponent (Clauset et al. 2009):

$$\hat{\gamma} = 1 + n\left[\sum_{i=1}^{n}\ln\frac{k_i}{k_{\min}}\right]^{-1}$$

The **branching ratio** $\sigma = \langle k_{\text{out}} \rangle$ measures the
mean number of direct aftershocks per event.  For ZBZ, $\sigma$ should be
clearly subcritical ($\sigma < 1$) because:
(a) background events have no parent — they contribute 0 to the in-degree count
but reduce the mean out-degree per node;
(b) the threshold removes low-weight links that inflated BP's $\sigma$ toward 1.

A subcritical $\sigma$ confirms that Italian seismicity is not self-exciting
at the catalog level — sequences die out rather than grow indefinitely.

*Reference:* Zaliapin et al. (2008) applied this analysis to California and
found $\sigma \approx 0.2$–$0.4$ after ZBZ declustering.

**Expected output:** $\hat{\gamma} \approx 2.0$–$2.5$; $\sigma \approx
0.2$–$0.6$ (subcritical, lower than BP's near-1 value due to thresholding);
CSN verdict likely "power law" with $p < 0.1$.

In [ ]:
out_degs  = [d for _, d in G.out_degree() if d > 0]
sigma_zbz = np.mean([d for _, d in G.out_degree()])

gamma_zbz = estimate_gamma_mle(out_degs, k_min=1)
print(f"Branching ratio σ = ⟨k_out⟩ = {sigma_zbz:.4f}  "
      f"({'subcritical' if sigma_zbz < 1 else 'supercritical'})")
print(f"MLE γ (out-degree, k_min=1): {gamma_zbz:.3f}")

result_zbz = test_power_law(out_degs, k_min=1)
print(f"  CSN test: γ={result_zbz['gamma']} (σ={result_zbz['sigma']})  "
      f"R={result_zbz['R']}  p={result_zbz['p_value']}  → {result_zbz['verdict']}")

## Preferential Attachment

The ZBZ forest is a GMM-thresholded version of the BP metric: only events
classified as *clustered* (below the GMM threshold on $\log_{10}\eta$) form
parent–child edges.  The empirical attachment kernel is:

$$\pi(k_{\text{out}}) = \frac{\sum_{i:\,k_i^{\text{out}}(t)=k} \Delta k_i}
{\#\{i:\,k_i^{\text{out}}(t)=k\}}$$

**$\alpha \approx 0$ is expected for the same reason as BP:** the parent is
still the event minimising $\eta_{ij}$ — a metric of time, distance, and
magnitude.  The out-degree of the candidate parent plays no role in the
selection, so a flat $\pi(k)$ is the theoretically correct prediction.

The value of measuring $\alpha_{ZBZ}$ alongside $\alpha_{BP}$ lies in
confirming that the GMM thresholding step does not accidentally introduce
degree dependence.  Both should yield $\alpha \approx 0$; agreement
validates that the ZBZ declustering preserves the same attachment
mechanism as the full BP forest.  The scale-free degree distribution in
both models originates from magnitude heterogeneity ($10^{-bm_i}$) via the
Gutenberg–Richter law, not from degree-dependent growth.

*Reference:* Jeong H., Néda Z. & Barabási A.-L. (2003). Measuring preferential
attachment in evolving networks. *Europhysics Letters*, 61(4), 567–572.

In [ ]:
print("Measuring preferential attachment kernel (ZBZ forest)…")
ks_pa, pi_k_pa, alpha_pa = measure_pa_forest(G, df_net)
plot_preferential_attachment(ks_pa, pi_k_pa, alpha_pa, title="ZBZ Italy (forest PA)")

## Macroscopic Metrics

In contrast to BP (one giant tree), the ZBZ forest contains many trees — one
per background event.  Key macroscopic properties:

- **Tree count**: equals the number of background events (roots).
- **Largest tree**: rooted at the most energetic mainshock; its size reflects
the spatial and temporal extent of the associated aftershock sequence.
Expected: largest tree ≈ 10,000–50,000 events (L'Aquila or Amatrice sequence).
- **Triggering-chain depth**: BFS from each tree's root measures the deepest
aftershock cascade.  ZBZ depths are expected to be **shorter** than BP depths
because the background-to-background bridges that create long BP paths are
removed.  Typical ZBZ max depth: 50–500 hops vs BP's thousands.
- **Average clustering** on the giant tree (undirected): ≈ 0 for a pure tree;
any nonzero value reflects numerical precision.

**Expected output:** giant tree contains 5–25% of catalog events; top-5 tree
sizes decline rapidly (power-law gap from largest to second-largest).
Depth histogram: right-skewed, max depth 50–500 hops, mean 5–20 hops.

In [ ]:
wcc_list    = list(nx.weakly_connected_components(G))
giant_wcc   = max(wcc_list, key=len)
G_giant     = G.subgraph(giant_wcc).copy()
G_und_giant = G_giant.to_undirected()

pct = G_giant.number_of_nodes() / G.number_of_nodes() * 100
print(f"Trees: {len(wcc_list):,}  |  Giant tree: {G_giant.number_of_nodes():,} nodes ({pct:.1f}%)")
print(f"Largest 5 tree sizes: {sorted([len(c) for c in wcc_list], reverse=True)[:5]}")

_root = next(n for n in G_giant.nodes() if G_giant.in_degree(n) == 0)
_depths = np.array(list(nx.single_source_shortest_path_length(G_giant, _root).values()))
print(f"\nGiant tree depth:  max={_depths.max()}  mean={_depths.mean():.2f}  "
      f"median={np.median(_depths):.0f}")

fig, ax = plt.subplots(figsize=(9, 4))
ax.hist(_depths, bins=50, color="steelblue", edgecolor="white", linewidth=0.4)
ax.set_xlabel("Depth from sequence root (hops)", fontsize=12)
ax.set_ylabel("Events", fontsize=12)
ax.set_title("Triggering-chain depth — ZBZ Italy (giant tree)", fontsize=13)
ax.set_yscale("log")
ax.grid(axis="y", ls="--", alpha=0.3)
ax.spines[["top", "right"]].set_visible(False)
plt.tight_layout()
savefig("zbz_depth_distribution_italy")
plt.show()

avg_c = nx.average_clustering(G_und_giant)
print(f"\nGiant tree (undirected): avg clustering = {avg_c:.4f}")

# Approximate avg path length from 500 random seeds within giant component
rng      = np.random.default_rng(42)
_gn      = list(giant_wcc)
_seeds   = rng.choice(_gn, size=min(500, len(_gn)), replace=False)
apl_vals = []
for s in _seeds:
    apl_vals.extend(nx.single_source_shortest_path_length(G_giant, s).values())
avg_L = np.mean(apl_vals)
print(f"Approx avg path length (giant tree, 500 seeds): {avg_L:.2f}")

## Centrality Analysis

## Centrality Analysis — 12 Measures

Twelve centrality measures are computed inline on the ZBZ declustered forest:

### Degree family
- **In-Degree**: 0 for background events (roots, no parent); 1 for all triggered
events.  In the ZBZ framework, zero in-degree directly labels *background seismicity*.
- **Out-Degree**: direct aftershock count — aftershock productivity of each event.
On the ZBZ forest, this is a cleaner productivity measure than in BP because
the GMM threshold removed background-to-background links.
- **Degree**: total connectivity (in + out).

### Path-based measures
- **PageRank**: long-run importance along aftershock chains.  On ZBZ, PR is
distributed across the roots of the largest aftershock families rather than
concentrating on the single chronological first event (as in BP).
- **Harmonic** $H_i = \sum_{j \neq i} 1/d_{ij}$: handles the disconnected
multi-tree forest; closeness = 0 for nodes in different weakly connected
components — harmonic is the correct generalisation.
- **Betweenness** ($k = 500$): "gateway" events on the trunk connecting two
large aftershock families; their removal would fragment the longest cascades.

### Spectral
- **Eigenvector / Katz**: importance weighted by neighbour importance.
Divergence from out-degree identifies events with unusually productive descendants.
- **HITS Hub / Authority**: Hub ≈ productivity; Authority ≈ being triggered by
many high-Hub sources.

### Local topology
- **Clustering / Triangles**: exactly 0 for a pure tree; any non-zero values
indicate that two background events in the same forest are also connected by
a second η-based link — a rare but theoretically possible ZBZ configuration.

In [ ]:
print("Computing centralities for ZBZ network…")
_t0_cent = time.time()

G_nsl     = G.copy()
G_nsl.remove_edges_from(nx.selfloop_edges(G_nsl))
G_und_all = G.to_undirected()
G_und_all.remove_edges_from(nx.selfloop_edges(G_und_all))
_N        = G.number_of_nodes()

in_deg_cent  = nx.in_degree_centrality(G)
out_deg_cent = nx.out_degree_centrality(G)
deg_cent     = nx.degree_centrality(G)
print(f"  Degree (in/out/total) done ({time.time()-_t0_cent:.1f}s)")

harm_cent = nx.harmonic_centrality(G)
print(f"  Harmonic done ({time.time()-_t0_cent:.1f}s)")

pr_cent = nx.pagerank(G, weight="weight")
print(f"  PageRank done ({time.time()-_t0_cent:.1f}s)")

bet_cent = nx.betweenness_centrality(G, k=min(500, _N), seed=42, normalized=True)
print(f"  Betweenness done ({time.time()-_t0_cent:.1f}s)")

try:
    eig_cent = nx.eigenvector_centrality(G_und_all, weight="weight", max_iter=500, tol=1e-6)
except nx.PowerIterationFailedConvergence:
    try:
        eig_cent = nx.eigenvector_centrality_numpy(G_und_all, weight="weight")
    except nx.AmbiguousSolution:
        # ZBZ is a disconnected forest — eigenvector centrality is undefined;
        # zero-fill and continue.
        eig_cent = {n: 0.0 for n in G.nodes()}
        print("  Eigenvector: skipped (graph is disconnected, undefined for forest)")
print(f"  Eigenvector done ({time.time()-_t0_cent:.1f}s)")

_max_deg    = max((G.degree(n) for n in G.nodes()), default=1)
_alpha_katz = 0.85 / _max_deg
try:
    katz_cent = nx.katz_centrality(
        G, alpha=_alpha_katz, weight="weight", normalized=True, max_iter=1000, tol=1e-6)
except nx.PowerIterationFailedConvergence:
    katz_cent = nx.katz_centrality_numpy(G, alpha=_alpha_katz, weight="weight")
print(f"  Katz done ({time.time()-_t0_cent:.1f}s)")

try:
    hits_hub, hits_auth = nx.hits(G_nsl, max_iter=1000, tol=1e-6)
except nx.PowerIterationFailedConvergence:
    _zeros    = {n: 0.0 for n in G.nodes()}
    hits_hub  = _zeros.copy()
    hits_auth = _zeros.copy()
print(f"  HITS done ({time.time()-_t0_cent:.1f}s)")

clust_cent = nx.clustering(G_und_all, weight="weight")
print(f"  Clustering done ({time.time()-_t0_cent:.1f}s)")
tri_count = nx.triangles(G_und_all)
print(f"  Triangles done ({time.time()-_t0_cent:.1f}s total)")

df_centrality = pd.DataFrame([
    {
        "cell_id":     n,
        "lat":         G.nodes[n]["lat"],
        "lon":         G.nodes[n]["lon"],
        "depth_km":    G.nodes[n]["depth_km"],
        "In_Degree":   in_deg_cent.get(n, 0.0),
        "Out_Degree":  out_deg_cent.get(n, 0.0),
        "Degree":      deg_cent.get(n, 0.0),
        "PageRank":    pr_cent.get(n, 0.0),
        "Harmonic":    harm_cent.get(n, 0.0),
        "Betweenness": bet_cent.get(n, 0.0),
        "Eigenvector": eig_cent.get(n, 0.0),
        "Katz":        katz_cent.get(n, 0.0),
        "HITS_Hub":    hits_hub.get(n, 0.0),
        "HITS_Auth":   hits_auth.get(n, 0.0),
        "Clustering":  clust_cent.get(n, 0.0),
        "Triangles":   float(tri_count.get(n, 0)),
    }
    for n in G.nodes()
    if "lat" in G.nodes[n] and "lon" in G.nodes[n]
])

for metric, label in [
    ("In_Degree",   "Background Events (In-Degree = 0)"),
    ("Out_Degree",  "Aftershock Productivity (Out-Degree)"),
    ("Degree",      "Most Productive Triggers"),
    ("PageRank",    "Long-Chain Influencers"),
    ("Harmonic",    "Topological Reach (Harmonic)"),
    ("Betweenness", "Sequence Bottlenecks"),
    ("HITS_Hub",    "Mainshock Hubs"),
    ("HITS_Auth",   "Aftershock Attractors"),
    ("Clustering",  "Multi-Path Events (Clustering)"),
    ("Triangles",   "Feedback Loops (Triangles)"),
]:
    print(f"\n--- Top 5: {label} ({metric}) ---")
    display(df_centrality.nlargest(5, metric)[
        ["cell_id", "lat", "lon", "depth_km", metric]])

plot_centrality_correlation(df_centrality, "ZBZ Italy")
plot_top_n_cells(df_centrality, top_n=10, title="ZBZ Italy")

## Centrality Geo Maps

Geographic projection of the top-10 events per centrality metric.  Two views:
per-metric interactive map (7 panels) and the overlap map (events coloured by
how many metrics they appear in simultaneously).

ZBZ centrality maps are expected to be **more spatially focused** than BP:
- BP's PageRank is dominated by the chronologically first event (the root of
the single spanning tree, which is the ancestor of all other events).
- ZBZ's PageRank is distributed across the roots of the largest aftershock
families — the most energetic background mainshocks over 40 years.

**Expected output:** top-10 events for degree/HITS Hub should coincide with
known major mainshocks (L'Aquila, Amatrice, Irpinia).  PageRank may highlight
a different set — events that are well-connected within the forest but not
necessarily the most productive individually (e.g., a sequence of moderate
events that collectively sustained a long cascade).

In [ ]:
plot_geo_top_n_interactive(
    df_centrality, top_n=10,
    title="ZBZ Italy",
    center_lat=41.9, center_lon=12.5, zoom=0, height=MAP_HEIGHT, width=MAP_WIDTH,
    bounds=_IT_BOUNDS,
)
plot_geo_centrality_overlap(
    df_centrality, top_n=10,
    title="ZBZ Italy",
    center_lat=41.9, center_lon=12.5, zoom=0, height=MAP_HEIGHT, width=MAP_WIDTH,
    bounds=_IT_BOUNDS,
)

## Bianconi-Barabasi Fitness

In the ZBZ forest, the GMM threshold separates background events (roots)
from clustered events (children).  The fitness estimator
$\hat{\beta}_i = \ln k_i(T) / \ln(T/t_i)$ is applied to all nodes with
$k > 1$ and $t_i > 0$, regardless of whether they are background or
clustered.  This allows comparison of the fitness distributions of the
two populations:

$$\hat{\beta}_i = \frac{\ln k_i(T)}{\ln(T/t_i)}$$

Background events (roots) with $k_{out} = 0$ and $k_{in} = 1$ are
excluded by the $k > 1$ criterion, focusing the analysis on events
that are genuinely productive mainshocks in the ZBZ sense.

Comparing the ZBZ fitness distribution with the BP fitness distribution
directly tests whether the thresholding step (removing background events)
changes the distribution of intrinsic seismogenic productivity.

Three theoretical regimes: equal-fitness BA, uniform fitness
($\hat{\beta} \sim U[0,\,\gamma-1]$), Bose-Einstein condensation.

*References:* Bianconi G. & Barabási A.-L. (2001). *EPL* 54, 436–442;
*PRL* 86, 5632–5635.

In [ ]:
print("Computing Bianconi-Barabasi fitness (ZBZ events)…")
df_bb = compute_bb_fitness_events(G, df_net)
print(f"  {len(df_bb)} events with valid β̂; β range [{df_bb['fitness_beta'].min():.3f}, {df_bb['fitness_beta'].max():.3f}]")
plot_bb_fitness(df_bb, title="ZBZ Italy")
plot_bb_fitness_theory(df_bb, gamma=gamma_zbz, title="ZBZ Italy")
plot_bb_fitness_geo(
    df_bb,
    title="ZBZ Italy",
    center_lat=41.9, center_lon=12.5, zoom=0,
    bounds=_IT_BOUNDS, height=MAP_HEIGHT, width=MAP_WIDTH,
)

## Community Detection — Full Suite

Community detection on the ZBZ forest is **seismologically the most meaningful**
of the event-level models: each weakly connected component is already a genuine
aftershock family by construction of the ZBZ threshold.  Seven algorithms are
compared, covering graph-based, density-based, and affiliation-based paradigms.

$$Q = \frac{1}{2m}\sum_{ij}\left[A_{ij} - \frac{k_i k_j}{2m}\right]\delta(c_i,c_j)$$

Louvain communities *within* the giant tree identify sub-sequences sharing a
common ancestral mainshock at a finer spatial/temporal scale; communities across
the full forest group distinct aftershock families that are topologically close in
the merged undirected graph.  Each ZBZ community should correspond to a
well-defined seismogenic zone, free of the spurious background-to-background
bridges that inflate BP modularity.

**Graph-based methods**
- **Louvain** (Blondel et al. 2008): greedy modularity maximisation on the
undirected forest; fast and scalable.
- **Consensus Louvain** (Lancichinetti & Fortunato 2012): 100-run ensemble
average; the most reproducible partition.
- **Spectral** (Von Luxburg 2007): restricted to the **undirected giant
component** to avoid a rank-deficient Laplacian from the disconnected multi-tree.
- **InfoMap** (directed; Rosvall & Bergstrom 2008): run on the **directed** ZBZ
graph, respecting the one-way parent→child triggering direction.

**Density-based methods**
- **HDBSCAN-Spectral** (Campello et al. 2013): applies HDBSCAN to the
$k$-dimensional spectral embedding of the normalised graph Laplacian of the
giant component.  No $k$ pre-specification is required; noise points are
excluded.  The embedding captures the multi-tree structure of the ZBZ forest
more faithfully than modularity-based methods.
- **HDBSCAN-Geographic**: same HDBSCAN algorithm applied to projected $(x, y)$
node coordinates in kilometres (EPSG:32632); no graph structure is used.
Comparing with HDBSCAN-Spectral tests whether ZBZ aftershock families are
primarily geographically co-located or whether the triggering topology encodes
additional structure beyond spatial proximity.

**Overlapping-community method**
- **BigCLAM** (Yang & Leskovec 2013): solves for an $N \times k$ affiliation
matrix $F$ via non-negative matrix factorisation; the hard partition is recovered
by $\arg\max_c F_{ic}$.  Events at fault intersections or spatially overlapping
aftershock zones may carry genuine dual affiliation in two ZBZ communities.

**Partition quality scoring**
All seven partitions are evaluated on 9 quality metrics — modularity, conductance,
coverage, Ncut, map equation, DC-SBM log-likelihood, Surprise, geographic
compactness, and depth coherence — via `compare_partitions`.

*References:* Newman & Girvan (2004) *Phys. Rev. E* 69, 026113;
Blondel et al. (2008) *J. Stat. Mech.* P10008;
Rosvall & Bergstrom (2008) *PNAS* 105, 1118;
Campello et al. (2013) *ECML-PKDD* 160–175;
Yang & Leskovec (2013) *ACM TKDD* 7(3), 1–42.

**Expected output:** Louvain finds 5–20 geographically compact communities; NMI
between Louvain and Consensus ≈ 0.4–0.7 (higher than BP because the cleaner forest
reduces algorithmic variance).  InfoMap on the directed ZBZ graph may find more
communities than on the undirected version, respecting the parent→child flow.
HDBSCAN-Geographic NMI with HDBSCAN-Spectral directly quantifies how much community
structure is geography-driven versus topology-driven in the ZBZ declustered forest.

In [ ]:
print("Building undirected ZBZ graph for community detection…")
G_und_comm = G.to_undirected()
G_und_comm.remove_edges_from(nx.selfloop_edges(G_und_comm))

print("Louvain…")
community_map = run_louvain(G_und_comm, seed=42)
k_louvain     = len(set(community_map.values()))
print(f"  {k_louvain} communities")
plot_community_geo(
    G_und_comm, community_map,
    title="ZBZ Italy",
    center_lat=41.9, center_lon=12.5, zoom=0, height=MAP_HEIGHT, width=MAP_WIDTH,
    bounds=_IT_BOUNDS,
    min_community_size=50, method_name="Louvain",
)

print("Consensus Louvain (100 runs)…")
community_map_consensus = run_consensus_louvain(G_und_comm, n_runs=100, seed=42)
print(f"  {len(set(community_map_consensus.values()))} communities")
plot_community_geo(
    G_und_comm, community_map_consensus,
    title="ZBZ Italy",
    center_lat=41.9, center_lon=12.5, zoom=0, height=MAP_HEIGHT, width=MAP_WIDTH,
    bounds=_IT_BOUNDS,
    min_community_size=50, method_name="Consensus Louvain",
)

print(f"Spectral (k={min(k_louvain, 8)}, giant component)…")
_k_spec = min(k_louvain, 8)
community_map_spectral = run_spectral(G_und_giant, k=_k_spec, seed=42)
print(f"  {len(set(community_map_spectral.values()))} communities (giant component only)")
plot_community_geo(
    G_und_giant, community_map_spectral,
    title="ZBZ Italy (giant component)",
    center_lat=41.9, center_lon=12.5, zoom=0, height=MAP_HEIGHT, width=MAP_WIDTH,
    bounds=_IT_BOUNDS,
    min_community_size=20, method_name="Spectral",
)

print("InfoMap (directed)…")
community_map_infomap = run_infomap(G, directed=True, seed=42)
print(f"  {len(set(community_map_infomap.values()))} communities")
plot_community_geo(
    G, community_map_infomap,
    title="ZBZ Italy",
    center_lat=41.9, center_lon=12.5, zoom=0, height=MAP_HEIGHT, width=MAP_WIDTH,
    bounds=_IT_BOUNDS,
    min_community_size=50, method_name="InfoMap (directed)",
)

_common_nodes = (set(community_map) & set(community_map_consensus)
                 & set(community_map_infomap))
partitions = {
    "Louvain":   {n: community_map[n]           for n in _common_nodes},
    "Consensus": {n: community_map_consensus[n] for n in _common_nodes},
    "InfoMap":   {n: community_map_infomap[n]   for n in _common_nodes},
}
print("Computing NMI…")
nmi_df = compute_nmi_matrix(partitions)
display(nmi_df.round(3))
plot_nmi_heatmap(nmi_df, title="ZBZ Italy")

print("HDBSCAN-Spectral…")
community_map_hdbscan_spec = run_hdbscan_spectral(G_und_giant, min_cluster_size=10, seed=42)
print(f"  {len(set(community_map_hdbscan_spec.values()))} clusters")
plot_community_geo(
    G_und_giant, community_map_hdbscan_spec,
    title="ZBZ Italy",
    center_lat=41.9, center_lon=12.5, zoom=0, height=MAP_HEIGHT, width=MAP_WIDTH,
    bounds=_IT_BOUNDS,
    min_community_size=20, method_name="HDBSCAN-Spectral",
)

print("HDBSCAN-Geographic…")
community_map_hdbscan_geo = run_hdbscan_geo(G_und_comm, min_cluster_size=10)
print(f"  {len(set(community_map_hdbscan_geo.values()))} clusters")
plot_community_geo(
    G_und_comm, community_map_hdbscan_geo,
    title="ZBZ Italy",
    center_lat=41.9, center_lon=12.5, zoom=0, height=MAP_HEIGHT, width=MAP_WIDTH,
    bounds=_IT_BOUNDS,
    min_community_size=20, method_name="HDBSCAN-Geographic",
)

print("BigCLAM…")
F_bigclam, community_map_bigclam = run_bigclam(
    G_und_comm, k=k_louvain, n_iter=100, lr=0.005, seed=42,
)
print(f"  {len(set(community_map_bigclam.values()))} non-empty communities")
plot_community_geo(
    G_und_comm, community_map_bigclam,
    title="ZBZ Italy",
    center_lat=41.9, center_lon=12.5, zoom=0, height=MAP_HEIGHT, width=MAP_WIDTH,
    bounds=_IT_BOUNDS,
    min_community_size=20, method_name="BigCLAM",
)

_hdbscan_spec_comm = {n: community_map_hdbscan_spec[n] for n in _common_nodes if n in community_map_hdbscan_spec}
_hdbscan_geo_comm  = {n: community_map_hdbscan_geo[n]  for n in _common_nodes if n in community_map_hdbscan_geo}
_bigclam_comm      = {n: community_map_bigclam[n]       for n in _common_nodes if n in community_map_bigclam}
_common_ext = set(_hdbscan_spec_comm) & set(_hdbscan_geo_comm) & set(_bigclam_comm)
partitions_ext = {
    **{k: {n: v[n] for n in _common_ext} for k, v in partitions.items()},
    "HDBSCAN-Spec": {n: _hdbscan_spec_comm[n] for n in _common_ext},
    "HDBSCAN-Geo":  {n: _hdbscan_geo_comm[n]  for n in _common_ext},
    "BigCLAM":      {n: _bigclam_comm[n]       for n in _common_ext},
}
nmi_ext = compute_nmi_matrix(partitions_ext)
display(nmi_ext.round(3))
plot_nmi_heatmap(nmi_ext, title="ZBZ Italy — all methods")

scores_df = compare_partitions(G_und_comm, partitions_ext, cell_size_km=10.0)
display(scores_df.round(4))
plot_partition_scores(scores_df, title="ZBZ Italy")

## Community Markov Flow

The Louvain partition coarse-grains the ZBZ forest into a $K \times K$ flow
matrix $F_{ij} = \sum_{u \in C_i,\, v \in C_j} w_{uv}$ (using the directed
ZBZ graph).  Row-normalisation gives the Markov transition matrix $T_{ij}$.

On the ZBZ forest $T_{ij}$ represents the probability that an aftershock chain
rooted in community $i$ crosses into community $j$ in the next hop — a direct
measure of **inter-family seismic stress transfer**.  Unlike BP, ZBZ flows are
restricted to genuine aftershock links, so high off-diagonal entries identify
true triggering corridors between distinct seismogenic zones rather than
background statistical correlations.

High self-retention $T_{ii} \to 1$ indicates communities are seismically
self-contained (internal aftershock chains dominate); low $T_{ii}$ identifies
communities that frequently pass seismic stress to neighbours — geologically
interesting zones with cross-fault triggering.

*Reference:* Rosvall, M., & Bergstrom, C. T. (2008). *PNAS*, 105(4), 1118–1123.

**Expected output:** $K$ = Louvain community count; mean self-retention
$T_{ii} \approx 0.7$–$0.95$; mean outflow entropy ≈ 1–3 bits.  The dominant
attractor $\pi_{\max}$ corresponds to the community hosting the densest
aftershock activity — typically the Apennine Central zone.

In [ ]:
from src.community_flow import (
    build_community_flow_matrix,
    compute_markov_chain,
    compute_stationary_distribution,
    community_flow_stats,
    plot_flow_heatmap,
    plot_flow_entropy,
    plot_stationary_distribution,
)

print("Building community flow matrix (Louvain, ZBZ network)…")
flow_count_df = build_community_flow_matrix(G, community_map)
T_markov  = compute_markov_chain(flow_count_df)
stat_dist = compute_stationary_distribution(T_markov)
df_flow   = community_flow_stats(T_markov, flow_count_df, stat_dist)

K_comm = T_markov.shape[0]
print(f"Markov chain: {K_comm} community states")
print(f"Mean self-retention:  {df_flow['self_retention'].mean():.3f}")
print(f"Mean outflow entropy: {df_flow['entropy'].mean():.3f} bits "
      f"(max = log₂({K_comm}) = {np.log2(K_comm):.3f})")
print(f"Dominant attractor:   Community {df_flow.iloc[0]['community']} "
      f"(π = {df_flow.iloc[0]['stationary']:.4f})")
display(df_flow)

plot_flow_heatmap(T_markov, flow_count_df, title="ZBZ Italy")
plot_flow_entropy(df_flow, K=K_comm, title="ZBZ Italy")
plot_stationary_distribution(df_flow, title="ZBZ Italy")

out_path = RESULTS_DIR / "data" / "italy_zbz_community_flow.csv"
df_flow.to_csv(out_path, index=False)
print(f"Community flow stats saved → {out_path}")

## Assortativity

Newman (2003) assortativity on the ZBZ forest.

**Magnitude assortativity** $r < 0$ (disassortative) is expected and physically
meaningful: background mainshocks (high $m_i$) trigger aftershocks (lower $m_j$)
— Båth's law as a network property.  The ZBZ threshold should produce a
**stronger** disassortative signal than BP because only genuine aftershock edges
are retained; background-to-background links (which can connect events of similar
magnitude) are removed.  Expect $r \approx -0.15$ to $-0.35$ (more negative
than BP's ≈ −0.1 to −0.2).

**Depth assortativity** $r \approx 0$: aftershock depths are not constrained by
the ZBZ metric (which uses epicentral distance, not depth) so depth correlation
reflects the raw seismotectonic structure.

**Degree assortativity** $r < 0$ (disassortative): productive background
mainshocks (high degree) connect to many leaf aftershocks (degree 1).

*Reference:* Newman, M. E. J. (2003). *Physical Review E*, 67(2), 026126.

**Expected output:** magnitude $r \approx -0.15$ to $-0.35$; degree
$r \approx -0.05$ to $-0.15$; depth $r \approx -0.1$ to $+0.1$.
Scatter panels with 2D histograms and regression lines.

In [ ]:
print("Computing assortativity (node attributes pre-attached)…")
df_assort = compute_assortativity(G)
display(df_assort)
plot_assortativity(G, title="ZBZ Italy")

print("Average nearest-neighbour degree k_nn(k):")
plot_knn(G, title="ZBZ Italy", gamma=gamma_zbz)

print("Rich-club coefficient:")
plot_rich_club(G, title="ZBZ Italy")

## Export Results

Three output files are written:

1. **CSV** (`italy_zbz_network_metrics.csv`) — one row per event with centrality
values, coordinates, depth, and Louvain community assignment.
2. **Pickle** (`italy_zbz.pkl`) — the full NetworkX directed ZBZ forest.
Reloads in ≈ 5 seconds for comparison notebooks.
3. **GEXF** (`italy_zbz.gexf`) — Gephi-compatible XML.  In Gephi, colour nodes
by community and use Yifan Hu layout to visualise the multi-tree structure.

**Expected output:** CSV ≈ 215,000 rows × 12 columns; pickle ≈ 100–250 MB
(sparser than BP because ZBZ has fewer edges); GEXF ≈ 50–150 MB.

In [ ]:
df_comm = pd.DataFrame(
    [{"cell_id": n, "community": community_map[n]} for n in community_map]
)
df_final = df_centrality.merge(
    df_comm[["cell_id", "community"]], on="cell_id", how="left"
)
out_path = RESULTS_DIR / "data" / "italy_zbz_network_metrics.csv"
df_final.to_csv(out_path, index=False)
print(f"Results saved → {out_path}  ({len(df_final):,} rows)")

pkl_path = RESULTS_DIR / "cache" / "italy_zbz.pkl"
with open(pkl_path, "wb") as f:
    pickle.dump(G, f)
print(f"Network cached → {pkl_path}")

gexf_path = RESULTS_DIR / "gephi" / "italy_zbz.gexf"
nx.write_gexf(G, gexf_path)
print(f"Gephi export → {gexf_path}")